# SMILES molecular generative models

A GRU variational autoencoder over character-tokenized SMILES. This demo trains briefly on the built-in tiny corpus to show the mechanics (tokenizer, reparameterization, autoregressive sampling, and validity/uniqueness/novelty metrics). The headline numbers in the README come from a longer run on 4188 public SMILES via `scripts/train.py`.

In [1]:
from molgen import (
    BUILTIN_SMILES, SmilesTokenizer, SmilesDataset, SmilesVAE, Trainer,
    generation_metrics, set_seed,
)

set_seed(0)
tok = SmilesTokenizer().build(BUILTIN_SMILES)
dataset = SmilesDataset(BUILTIN_SMILES, tok)
print('vocab size:', len(tok), '| training molecules:', len(dataset))

vocab size: 15 | training molecules: 20


In [2]:
model = SmilesVAE(len(tok), pad_id=tok.pad_id)
trainer = Trainer(model, pad_id=tok.pad_id, lr=1e-3, max_beta=0.1, kl_anneal_epochs=10)
trainer.fit(dataset, epochs=20, batch_size=32, verbose=False)
print('final loss:', round(trainer.history['loss'][-1], 4))

final loss: 1.4384


In [3]:
gen_ids = model.generate(200, tok.sos_id, tok.eos_id, temperature=0.9)
gen = [tok.decode(ids) for ids in gen_ids]
metrics = generation_metrics(gen, BUILTIN_SMILES)
print({k: round(v, 3) for k, v in metrics.items()})

{'n_generated': 200.0, 'validity': 0.145, 'uniqueness': 0.724, 'novelty': 0.905}
